In [26]:
import numpy as np
from scipy import sparse

import gurobipy as gp
from gurobipy import GRB
from scipy.sparse.linalg import lsmr, lsqr
import time

In [ ]:
def generate_quantile_instance(
    m: int,
    p: int,
    density: float = 0.8,
    seed: int | None = None,
):
    """
    Generate one synthetic Quantile Regression instance.

    Parameters
    ----------
    m : int
        Number of observations / records.

    p : int
        Number of explanatory variables (columns of D).

    density : float, default=0.8
        Target proportion of nonzero entries in D.

    seed : int or None, default=None
        Random seed for reproducibility.

    Returns
    -------
    D : ndarray, shape (m, p)
        Design matrix.

    b : ndarray, shape (m,)
        Dependent variable.
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # check input values
    if m <= 0:
        raise ValueError("m must be positive")

    if p <= 0:
        raise ValueError("p must be positive")

    if not 0.0 <= density <= 1.0:
        raise ValueError("density must lie between 0 and 1")

    # initialize D as an all-zero matrix
    D = np.zeros((m, p), dtype=float)

    # calculate the total number of entries
    n_total = m * p

    # calculate the number of nonzero entries
    n_nonzero = int(round(density * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # fill the selected positions with values sampled from U[-1, 1]
    D.flat[nonzero_indices] = rng.uniform(-1.0, 1.0, size=n_nonzero)

    # generate the dependent variable from U[-1, 1]
    b = rng.uniform(-1.0, 1.0, size=m)

    return D, b

In [ ]:
# generate one quantile regression instance
D, b = generate_quantile_instance(m=5000, p=399, density=0.8, seed=42)

# check dimensions
print("D shape:", D.shape)
print("b shape:", b.shape)

# check density
actual_density = np.count_nonzero(D) / D.size

print("number of entries in D:", D.size)
print("number of nonzero entries:", np.count_nonzero(D))
print("actual density:", actual_density)

# check value ranges
print("D min:", D.min())
print("D max:", D.max())
print("b min:", b.min())
print("b max:", b.max())

D shape: (5000, 399)
b shape: (5000,)
number of entries in D: 1995000
number of nonzero entries: 1596000
actual density: 0.8
D min: -0.9999928141121515
D max: 0.9999998390380078
b min: -0.9994594203105618
b max: 0.9998598619646724


In [ ]:
def build_quantile_lp(
    D: np.ndarray,
    b: np.ndarray,
    tau: float = 0.2,
):
    """
    Build the LP formulation of a quantile regression instance.

    Parameters
    ----------
    D : ndarray, shape (m, p)
        Design matrix

    b : ndarray, shape (m,)
        Dependent variable

    tau : float, default=0.2
        Quantile level

    Returns
    -------
    A : sparse matrix, shape (m, p + 2m)
        Equality constraint matrix

    c : ndarray, shape (p + 2m,)
        Objective coefficient vector

    bounds : list
        Variable bounds for beta, u_plus, and u_minus
    """

    # get the dimensions of D
    m, p = D.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the quantile level
    if not 0.0 < tau < 1.0:
        raise ValueError("tau must lie strictly between 0 and 1")

    # convert D to sparse format
    D_sparse = sparse.csr_matrix(D)

    # construct a sparse identity matrix
    I = sparse.eye(m, format="csr")

    # construct A = [D | I | -I]
    A = sparse.hstack((D_sparse, I, -I), format="csr")

    # construct the objective coefficient vector
    c = np.concatenate(
        (np.zeros(p), tau * np.ones(m), (1.0 - tau) * np.ones(m))
    )

    return A, c

In [ ]:
# build the quantile regression LP
A, c = build_quantile_lp(D, b, tau=0.2)

# check dimensions
print("A shape:", A.shape)
print("c shape:", c.shape)

# check the three blocks of the objective
print("beta coefficients:", np.unique(c[: D.shape[1]]))
print(
    "u_plus coefficients:", np.unique(c[D.shape[1] : D.shape[1] + D.shape[0]])
)
print("u_minus coefficients:", np.unique(c[D.shape[1] + D.shape[0] :]))

A shape: (5000, 10399)
c shape: (10399,)
beta coefficients: [0.]
u_plus coefficients: [0.2]
u_minus coefficients: [0.8]


In [ ]:
def generate_projection_matrix(
    A,
    k: int,
    seed: int | None = None,
):
    """
    Generate a sparse random projection matrix T.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Constraint matrix

    k : int
        Projected dimension

    seed : int or None, default=None
        Random seed for reproducibility

    Returns
    -------
    T : sparse matrix, shape (k, m)
        Sparse random projection matrix
    """

    # create a local random number generator
    rng = np.random.default_rng(seed)

    # get the number of original constraints
    m = A.shape[0]

    # check the projected dimension
    if not 0 < k < m:
        raise ValueError("k must lie strictly between 0 and m")

    # calculate the density of A
    density_A = A.nnz / (A.shape[0] * A.shape[1])

    # set the density of T
    sigma = density_A / 2.0

    # calculate the total number of entries in T
    n_total = k * m

    # calculate the number of nonzero entries in T
    n_nonzero = int(round(sigma * n_total))

    # randomly choose the positions of nonzero entries
    nonzero_indices = rng.choice(n_total, size=n_nonzero, replace=False)

    # convert the flat indices to row and column indices
    rows = nonzero_indices // m
    cols = nonzero_indices % m

    # calculate the standard deviation of nonzero entries
    std = 1.0 / np.sqrt(k * sigma)

    # sample the nonzero entries from the Gaussian distribution
    values = rng.normal(loc=0.0, scale=std, size=n_nonzero)

    # construct the sparse projection matrix
    T = sparse.csr_matrix((values, (rows, cols)), shape=(k, m))

    return T

In [ ]:
# generate a sparse random projection matrix
start_time = time.perf_counter()
T = generate_projection_matrix(A, k=1000, seed=42)
time_T_generation = time.perf_counter() - start_time

# check dimensions
print("T shape:", T.shape)

# check the number of nonzero entries
print("T nnz:", T.nnz)

# check the density of A
density_A = A.nnz / (A.shape[0] * A.shape[1])
print("A density:", density_A)

# check the target density of T
sigma = density_A / 2.0
print("target T density:", sigma)

# check the actual density of T
density_T = T.nnz / (T.shape[0] * T.shape[1])
print("actual T density:", density_T)

T shape: (1000, 5000)
T nnz: 77219
A density: 0.030887585344744686
target T density: 0.015443792672372343
actual T density: 0.0154438


In [33]:
def project_lp(
    A,
    b: np.ndarray,
    T,
):
    """
    Project the equality constraints of an LP.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Original constraint matrix

    b : ndarray, shape (m,)
        Original right-hand side vector

    T : sparse matrix, shape (k, m)
        Random projection matrix

    Returns
    -------
    A_projected : sparse matrix, shape (k, n)
        Projected constraint matrix

    b_projected : ndarray, shape (k,)
        Projected right-hand side vector
    """

    # check the dimensions of A and b
    if b.shape != (A.shape[0],):
        raise ValueError("b must have shape (m,)")

    # check the dimensions of T and A
    if T.shape[1] != A.shape[0]:
        raise ValueError("T and A have incompatible dimensions")

    # project the constraint matrix
    A_projected = T @ A

    # project the right-hand side vector
    b_projected = T @ b

    return A_projected, b_projected

In [ ]:
def solve_lp(
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Solve a linear program using Gurobi.

    Parameters
    ----------
    A : sparse matrix, shape (m, n)
        Equality constraint matrix

    b : ndarray, shape (m,)
        Right-hand side vector

    c : ndarray, shape (n,)
        Objective coefficient vector

    lb : ndarray, shape (n,)
        Lower bounds of the decision variables

    ub : ndarray, shape (n,)
        Upper bounds of the decision variables

    Returns
    -------
    x_opt : ndarray, shape (n,)
        Optimal solution

    objective_value : float
        Optimal objective value
    """

    # get the number of constraints and decision variables
    m, n = A.shape

    # check the dimension of b
    if b.shape != (m,):
        raise ValueError("b must have shape (m,)")

    # check the dimension of c
    if c.shape != (n,):
        raise ValueError("c must have shape (n,)")

    # check the dimension of the lower bounds
    if lb.shape != (n,):
        raise ValueError("lb must have shape (n,)")

    # check the dimension of the upper bounds
    if ub.shape != (n,):
        raise ValueError("ub must have shape (n,)")

    # create a Gurobi model
    model = gp.Model("lp_model")

    # create the decision variables
    x = model.addMVar(shape=n, lb=lb, ub=ub, name="x")

    # set the objective function
    model.setObjective(c @ x, GRB.MINIMIZE)

    # add the equality constraints
    model.addConstr(A @ x == b)

    # solve the LP
    model.optimize()

    # check the optimization status
    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(
            f"Gurobi did not find an optimal solution, status={model.Status}"
        )

    # extract the optimal solution
    x_opt = x.X

    # extract the optimal objective value
    objective_value = model.ObjVal

    return x_opt, objective_value

In [ ]:
def build_quantile_bounds(
    m: int,
    p: int,
):
    """
    Build the variable bounds for the quantile regression LP.

    Parameters
    ----------
    m : int
        Number of observations

    p : int
        Number of regression coefficients

    Returns
    -------
    lb : ndarray, shape (p + 2m,)
        Lower bounds

    ub : ndarray, shape (p + 2m,)
        Upper bounds
    """

    # construct the lower bounds
    lb = np.concatenate((np.full(p, -np.inf), np.zeros(m), np.zeros(m)))

    # construct the upper bounds
    ub = np.full(p + 2 * m, GRB.INFINITY)

    return lb, ub

In [ ]:
# project the original equality constraints
start_time = time.perf_counter()
A_projected, b_projected = project_lp(A, b, T)
time_projection = time.perf_counter() - start_time

# get the dimensions of the quantile regression instance
m, p = D.shape

lb, ub = build_quantile_bounds(m=m, p=p)

# solve the projected LP (including Gurobi model setup)
start_time = time.perf_counter()
x_tilde, obj_tilde = solve_lp(A_projected, b_projected, c, lb, ub)
time_TLP_solve = time.perf_counter() - start_time

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1000 rows, 10399 columns and 553438 nonzeros (Min)
Model fingerprint: 0x5be11a32
Model has 10000 linear objective coefficients
Coefficient statistics:
  Matrix range     [5e-06, 6e+00]
  Objective range  [2e-01, 8e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-03, 4e+00]

Presolve time: 0.44s
Presolved: 1000 rows, 10399 columns, 553438 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.02s

Barrier statistics:
 Dense cols : 399
 Free vars  : 399
 AA' NZ     : 7.471e+05
 Factor NZ  : 9.793e+05 (roughly 12 MB of memory)
 Factor Ops : 9.137e+08 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual

In [ ]:
def solve_projection_system(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    x_check: np.ndarray,
):
    """
    Approximate the projection of x0 onto A x = b using LSMR.
    """

    # solve A x_hat = b + A x0
    x_hat = lsmr(A, b + A @ x0)[0]

    # construct a second point approximately satisfying A x = b
    x_prime = x_hat - x0

    # construct the direction from x_prime to x_check
    direction = x_check - x_prime

    # handle the degenerate case
    if direction @ direction == 0.0:
        return x_prime

    # calculate the projection coefficient
    lambda_value = (direction @ (x0 - x_prime)) / (direction @ direction)

    # project x0 onto the line through x_prime and x_check
    x_dagger = x_prime + lambda_value * direction

    return x_dagger

In [ ]:
def retrieval_feasibility(x, A, b, lb, ub, eq_tol=0.01, bound_tol=1e-8):
    """Check BOTH original constraints; iterate change alone is not feasibility."""
    eq_residual = float(np.linalg.norm(A @ x - b))
    lower_error = np.maximum(lb - x, 0.0)
    finite_upper = np.isfinite(ub)
    upper_error = np.zeros_like(x)
    upper_error[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )
    bound_error = float(max(np.max(lower_error), np.max(upper_error)))
    return eq_residual <= eq_tol and bound_error <= bound_tol


def retrieve_apm(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """Alternate affine and bound projections. Return (point, steps, reason)."""
    x = x0.copy()
    x_check = lsmr(A, b)[0]

    for iteration in range(1, max_iter + 1):
        x_dagger = solve_projection_system(x, A, b, x_check)
        if retrieval_feasibility(x_dagger, A, b, lb, ub, eq_tol=delta):
            return x_dagger, iteration, "feasible (affine projection)"

        x_new = np.clip(x_dagger, lb, ub)
        if retrieval_feasibility(x_new, A, b, lb, ub, eq_tol=delta):
            return x_new, iteration, "feasible (bound projection)"
        if np.linalg.norm(x_new - x) < delta:
            return x_new, iteration, "stagnated; check residuals"
        x = x_new

    return x, max_iter, "iteration limit; check residuals"

In [ ]:
def evaluate_retrieval(
    x: np.ndarray,
    A,
    b: np.ndarray,
    c: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
):
    """
    Evaluate a retrieved solution.
    """

    # calculate the objective value
    objective_value = c @ x

    # calculate the equality constraint residual
    equality_residual = np.linalg.norm(A @ x - b)

    # calculate lower-bound violations
    lower_violation = np.maximum(lb - x, 0.0)

    # calculate upper-bound violations
    upper_violation = np.zeros_like(x)

    finite_upper = np.isfinite(ub)

    upper_violation[finite_upper] = np.maximum(
        x[finite_upper] - ub[finite_upper], 0.0
    )

    # calculate the maximum bound violation
    bound_violation = max(np.max(lower_violation), np.max(upper_violation))

    return (objective_value, equality_residual, bound_violation)

In [ ]:
# solve the original LP (including Gurobi model setup)
start_time = time.perf_counter()
x_original, obj_original = solve_lp(A, b, c, lb, ub)
time_original_solve = time.perf_counter() - start_time

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (mac64[arm] - Darwin 24.6.0 24G90)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 5000 rows, 10399 columns and 1606000 nonzeros (Min)
Model fingerprint: 0x37300496
Model has 10000 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-07, 1e+00]
  Objective range  [2e-01, 8e-01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-04, 1e+00]

Presolve time: 0.42s
Presolved: 5000 rows, 10399 columns, 1606000 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.06s

Barrier statistics:
 Dense cols : 399
 Free vars  : 399
 AA' NZ     : 1.596e+06
 Factor NZ  : 1.681e+06 (roughly 26 MB of memory)
 Factor Ops : 5.342e+08 (less than 1 second per iteration)
 Threads    : 8

                  Objective                Residual
Iter       Primal          Dual         Primal    Du

In [ ]:
start_time = time.perf_counter()
x_apm, apm_iterations, apm_status = retrieve_apm(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=30, delta=0.01
)

time_apm_retrieval = time.perf_counter() - start_time

(
    apm_objective,
    apm_residual,
    apm_bound_violation,
) = evaluate_retrieval(
    x_apm,
    A,
    b,
    c,
    lb,
    ub,
)

print("APM objective:", apm_objective)
print("APM equality residual:", apm_residual)
print("APM bound violation:", apm_bound_violation)

APM objective: 1360.501386757206
APM equality residual: 0.008920022794710757
APM bound violation: 0.0


In [ ]:
if abs(obj_original) > 1e-12:
    relative_objective_error = abs(apm_objective - obj_original) / abs(
        obj_original
    )
    print("Relative objective error:", relative_objective_error)
else:
    relative_objective_error = np.nan
    print("Relative objective error: undefined because obj_original is 0")

Relative objective error: 0.21144802109770955


In [ ]:
def retrieve_dykstra(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
):
    """Dykstra: affine set first, bounds second; check both at return."""
    x = x0.copy()
    x_check = lsmr(A, b)[0]
    p = np.zeros_like(x)  # affine correction
    q = np.zeros_like(x)  # bound correction

    for iteration in range(1, max_iter + 1):
        affine_input = x + p
        y = solve_projection_system(affine_input, A, b, x_check)
        p = affine_input - y

        bounds_input = y + q
        x_new = np.clip(bounds_input, lb, ub)
        q = bounds_input - x_new

        if retrieval_feasibility(x_new, A, b, lb, ub, eq_tol=delta):
            return x_new, iteration, "feasible"
        if np.linalg.norm(x_new - x) < delta:
            return x_new, iteration, "stagnated; check residuals"
        x = x_new

    return x, max_iter, "iteration limit; check residuals"

In [ ]:
start_time = time.perf_counter()
x_dykstra, dykstra_iterations, dykstra_status = retrieve_dykstra(
    x0=x_tilde, A=A, b=b, lb=lb, ub=ub, max_iter=30, delta=0.01
)

time_dykstra_retrieval = time.perf_counter() - start_time

In [ ]:
dykstra_objective, dykstra_residual, dykstra_bound_violation = (
    evaluate_retrieval(x_dykstra, A, b, c, lb, ub)
)

print("Dykstra objective:", dykstra_objective)
print("Dykstra equality residual:", dykstra_residual)
print("Dykstra bound violation:", dykstra_bound_violation)

Dykstra objective: 1359.2002539973437
Dykstra equality residual: 0.009470132492496671
Dykstra bound violation: 0.0


In [46]:
print("===== von Neumann APM =====")
print("Iterations:", apm_iterations)
print("Objective:", apm_objective)
print("Equality residual:", apm_residual)
print("Bound violation:", apm_bound_violation)

print("\n===== Dykstra APM =====")
print("Iterations:", dykstra_iterations)
print("Objective:", dykstra_objective)
print("Equality residual:", dykstra_residual)
print("Bound violation:", dykstra_bound_violation)

===== von Neumann APM =====
Iterations: 12
Objective: 1360.501386757206
Equality residual: 0.008920022794710757
Bound violation: 0.0

===== Dykstra APM =====
Iterations: 12
Objective: 1359.2002539973437
Equality residual: 0.009470132492496671
Bound violation: 0.0


In [ ]:
def retrieve_aarm(
    x0: np.ndarray,
    A,
    b: np.ndarray,
    lb: np.ndarray,
    ub: np.ndarray,
    max_iter: int = 30,
    delta: float = 0.01,
    alpha: float = 0.9,
    beta: float = 0.9,
):
    """Apply AARM; convergence of the operator does not certify LP feasibility."""
    x = x0.copy()
    x_check = lsmr(A, b)[0]

    for iteration in range(1, max_iter + 1):
        x_u = solve_projection_system(x, A, b, x_check)
        r_u = 2 * beta * x_u - x
        x_s = np.clip(r_u, lb, ub)
        r_s = 2 * beta * x_s - r_u
        x_new = (1 - alpha) * x + alpha * r_s

        if retrieval_feasibility(x_new, A, b, lb, ub, eq_tol=delta):
            return x_new, iteration, "feasible"
        if np.linalg.norm(x_new - x) < delta:
            return x_new, iteration, "stagnated; check residuals"
        x = x_new

    return x, max_iter, "iteration limit; check residuals"

In [ ]:
start_time = time.perf_counter()
x_aarm, aarm_iterations, aarm_status = retrieve_aarm(
    x0=x_tilde,
    A=A,
    b=b,
    lb=lb,
    ub=ub,
    max_iter=30,
    delta=0.01,
    alpha=0.9,
    beta=0.9,
)

time_aarm_retrieval = time.perf_counter() - start_time

print("AARM iterations:", aarm_iterations)
print("AARM stop reason:", aarm_status)

AARM iterations: 26
AARM stop reason: stagnated; check residuals


In [ ]:
aarm_objective, aarm_residual, aarm_bound_violation = evaluate_retrieval(
    x_aarm, A, b, c, lb, ub
)

print("AARM objective:", aarm_objective)
print("AARM equality residual:", aarm_residual)
print("AARM bound violation:", aarm_bound_violation)

AARM objective: 1179.5446675964777
AARM equality residual: 15.726794126055713
AARM bound violation: 0.0


In [ ]:
# Compare objectives only when ORIGINAL constraints are satisfied.
# The TLP objective is for a different, projected feasible set.
print("===== Original LP =====")
print("Objective:", obj_original)
print("\n===== TLP (projected LP) =====")
print("Objective:", obj_tilde)

for name, x, steps, status, objective, eq_residual, bound_error in [
    (
        "von Neumann APM",
        x_apm,
        apm_iterations,
        apm_status,
        apm_objective,
        apm_residual,
        apm_bound_violation,
    ),
    (
        "Dykstra APM",
        x_dykstra,
        dykstra_iterations,
        dykstra_status,
        dykstra_objective,
        dykstra_residual,
        dykstra_bound_violation,
    ),
    (
        "AARM",
        x_aarm,
        aarm_iterations,
        aarm_status,
        aarm_objective,
        aarm_residual,
        aarm_bound_violation,
    ),
]:
    feasible = retrieval_feasibility(x, A, b, lb, ub, eq_tol=0.01)
    print(f"\n===== {name} =====")
    print("Iterations:", steps)
    print("Stop reason:", status)
    print("Original LP feasible at tolerance:", feasible)
    print(
        "Objective:",
        objective,
        "(comparable)" if feasible else "(infeasible point)",
    )
    print("Equality residual (L2):", eq_residual)
    print("Maximum bound violation:", bound_error)

# Wall-clock times for the same single problem instance.
# Shared data generation (D, b, A) is excluded from both pipelines.
print("\n===== Wall-clock time (seconds) =====")
print(f"Generate T:           {time_T_generation:.3f}")
print(f"Compute TA and Tb:    {time_projection:.3f}")
print(f"Solve TLP:            {time_TLP_solve:.3f}")
print(f"Solve original LP:    {time_original_solve:.3f}")
print(f"von Neumann retrieval:{time_apm_retrieval:.3f}")
print(f"Dykstra retrieval:    {time_dykstra_retrieval:.3f}")
print(f"AARM retrieval:       {time_aarm_retrieval:.3f}")

# Compare complete RP + retrieval workflows with the direct original solve.
time_projected_base = time_T_generation + time_projection + time_TLP_solve
print("\n===== Complete pipelines (seconds) =====")
print(f"Original LP directly:             {time_original_solve:.3f}")
print(
    f"RP + TLP + von Neumann retrieval: {time_projected_base + time_apm_retrieval:.3f}"
)
print(
    f"RP + TLP + Dykstra retrieval:     {time_projected_base + time_dykstra_retrieval:.3f}"
)
print(
    f"RP + TLP + AARM retrieval:        {time_projected_base + time_aarm_retrieval:.3f} (check feasibility above)"
)

# A ratio above 1 means the Dykstra pipeline ran faster than a direct LP solve.
dykstra_pipeline_time = time_projected_base + time_dykstra_retrieval
print(
    f"Direct / Dykstra pipeline:        {time_original_solve / dykstra_pipeline_time:.3f}x"
)

===== Original LP =====
Objective: 1123.0373594769978

===== TLP (projected LP) =====
Objective: 121.12974575340694

===== von Neumann APM =====
Iterations: 12
Stop reason: feasible (bound projection)
Original LP feasible at tolerance: True
Objective: 1360.501386757206 (comparable)
Equality residual (L2): 0.008920022794710757
Maximum bound violation: 0.0

===== Dykstra APM =====
Iterations: 12
Stop reason: feasible
Original LP feasible at tolerance: True
Objective: 1359.2002539973437 (comparable)
Equality residual (L2): 0.009470132492496671
Maximum bound violation: 0.0

===== AARM =====
Iterations: 26
Stop reason: stagnated; check residuals
Original LP feasible at tolerance: False
Objective: 1179.5446675964777 (infeasible point)
Equality residual (L2): 15.726794126055713
Maximum bound violation: 0.0

===== Wall-clock time (seconds) =====
Generate T:           0.005
Compute TA and Tb:    0.039
Solve TLP:            1.734
Solve original LP:    2.091
von Neumann retrieval:0.721
Dykstra re